In [1]:
print("ok")

ok


In [2]:
%pwd

'c:\\Users\\Anshul Parkar\\Documents\\GitHub\\End-To-End_GenAI_ChatBot\\research'

In [3]:
import os
os.chdir("../")

In [4]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\Anshul Parkar\.conda\envs\venv_py311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
#Extract Data From the PDF File
def load_pdf_file(data):
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)

    documents=loader.load()

    return documents

In [7]:
extracted_data=load_pdf_file(data='Data/')

In [8]:
# extracted_data

In [9]:
#Split the Data into Text Chunks
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [10]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 792


In [11]:
# text_chunks

In [12]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [13]:
# %pip install -U sentence-transformers


In [14]:
#Download the Embeddings from Hugging Face\
# from sentence_transformers import SentenceTransformer
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [15]:
# %pip install sentence_transformers

In [16]:
# from sentence_transformers import SentenceTransformer
embeddings = download_hugging_face_embeddings()

C:\Users\Anshul Parkar\AppData\Local\Temp\ipykernel_35180\1553310711.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')


In [17]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [18]:
query_result

[-0.03447727486491203,
 0.03102312609553337,
 0.006734980270266533,
 0.026108933612704277,
 -0.03936205804347992,
 -0.16030246019363403,
 0.06692394614219666,
 -0.006441438104957342,
 -0.047450482845306396,
 0.014758863486349583,
 0.07087534666061401,
 0.05552757531404495,
 0.019193356856703758,
 -0.02625126577913761,
 -0.01010954286903143,
 -0.026940442621707916,
 0.022307462990283966,
 -0.02222665585577488,
 -0.14969263970851898,
 -0.017493024468421936,
 0.007676282897591591,
 0.054352231323719025,
 0.0032544038258492947,
 0.03172588348388672,
 -0.08462139964103699,
 -0.029405992478132248,
 0.051595550030469894,
 0.048124078661203384,
 -0.003314835485070944,
 -0.05827915295958519,
 0.04196925833821297,
 0.022210702300071716,
 0.1281888633966446,
 -0.022338951006531715,
 -0.011656239628791809,
 0.06292837113142014,
 -0.03287634998559952,
 -0.09122604131698608,
 -0.03117534890770912,
 0.052699536085128784,
 0.04703483358025551,
 -0.08420310169458389,
 -0.030056182295084,
 -0.0207448396

In [19]:
from dotenv import load_dotenv
load_dotenv()

True

In [20]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
GROQ_API_KEY=os.environ.get('GROQ_API_KEY')

In [21]:
# %pip install pinecone

In [22]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "healthai"

# Check if index already exists before creating
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384, 
        metric="cosine", 
        spec=ServerlessSpec(
            cloud="aws", 
            region="us-east-1"
        ) 
    )
    print(f"Index '{index_name}' created successfully!")
else:
    print(f"Index '{index_name}' already exists.")

# Connect to the index
index = pc.Index(index_name)

Index 'healthai' already exists.


In [23]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [24]:
# Embed each chunk and upsert the embeddings into your Pinecone index.
from langchain_pinecone import Pinecone as PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings
)

In [25]:
# Load Existing index 

from langchain_pinecone import Pinecone as PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [26]:
docsearch

In [27]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [28]:
retrieved_docs = retriever.invoke("What is Mental Health Worldwide?")

In [29]:
retrieved_docs

[Document(metadata={'author': 'pam smith', 'creationdate': '2014-01-14T15:39:55-05:00', 'creator': 'Microsoft® Office Publisher 2007', 'gts_pdfxconformance': 'PDF/X-1a:2001', 'gts_pdfxversion': 'PDF/X-1:2001', 'moddate': '2014-01-14T16:06:51-05:00', 'page': 11.0, 'page_label': '12', 'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'source': 'Data\\MentalHealthGuideBook.pdf', 'title': 'merged.pdf', 'total_pages': 261.0, 'trapped': '/False'}, page_content='I: Mental Health Worldwide \n11 \nPART I: MENTAL HEALTH WORLDWIDE \nThe World Health Organization (WHO) has defined mental health \nas a “state of well-being in which every individual realizes his or \nher own potential, can cope with the normal stresses of life, can \nwork productively and fruitfully, and is able to make a contribution \nto her or his community.”   \nGlobal mental health refers to the international perspective on \nvaried aspects of mental health and has been defined as “the area of'),
 Document(metadata={'author': '

In [30]:
# %pip install langchain-groq

In [31]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",  # or "llama2-70b-4096", "gemma-7b-it"
    temperature=0.4,
    max_tokens=500,
    api_key=GROQ_API_KEY
)

In [32]:
# Install required package if missing
# %pip install langchain-core langchain


In [33]:
# %pip install -U langchain langchain-community

In [34]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# Query
question = "What is Mental Health Worldwide?"

# Retrieve relevant documents
retrieved_docs = retriever.invoke(question)

# Format context from retrieved documents
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

# Create the final prompt with context
messages = prompt.format_messages(context=context, input=question)

# Get response from LLM
response = llm.invoke(messages)

# Print the answer
print("Question:", question)
print("\nAnswer:", response.content)

# Optional: Show source documents
print("\n" + "="*50)
print("Source Documents:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n{i}. {doc.page_content[:200]}...")

Question: What is Mental Health Worldwide?

Answer: Mental Health Worldwide refers to the international perspective on various aspects of mental health. It is defined as the area of study that examines mental health issues across different cultures, countries, and populations.

Source Documents:

1. I: Mental Health Worldwide 
11 
PART I: MENTAL HEALTH WORLDWIDE 
The World Health Organization (WHO) has defined mental health 
as a “state of well-being in which every individual realizes his or 
her...

2. References 
229 
PART I: Mental Health Worldwide 
Ghodse H. International Perspectives on Mental Health. London: 
Royal College of Psychiatrists; 2011. <http://public.eblib.com/
EBLPublic/PublicView.d...

3. I: Mental Health Worldwide 
13 
Information presented in this section comes from reports and 
studies that have inherent limitations, therefore specific conclu-
sions and summary statements will have ...


In [35]:
# %pip install -U langchain-core langchain

In [36]:
question_answer_chain = llm | prompt

In [37]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# Helper function to format documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create the RAG chain using LCEL (LangChain Expression Language)
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Test the chain
response = rag_chain.invoke("what is Mental Health?")
print("Answer:", response)

Answer: The World Health Organization (WHO) defines mental health as a "state of well-being in which every individual realizes his or her own potential, can cope with the normal stresses of life, can work productively and fruitfully, and is able to make a contribution to her or his community."


In [38]:
response = rag_chain.invoke("what is Conditions & Issues: Medication Guide?")
print("Answer:", response)

Answer: The "Conditions & Issues: Medication Guide" is a section in a manual that provides information on medications used to treat mental health conditions. It is intended for authorized prescribers, such as physicians and nurse practitioners, to help them make informed decisions about medication therapy.


In [39]:
response = rag_chain.invoke("Special notes regarding antipsychotic medication ")
print("Answer:", response)

Answer: Atypical antipsychotics are associated with the metabolic syndrome and increased mortality in elderly patients with dementia-related psychosis.


In [ ]:
response = rag_chain.invoke("How is Anshul ")
print("Answer:", response)

To check vector index status, visit:

https://app.pinecone.io/organizations/-Oc9LNv6C1sbdTNlkWs4/projects/7662a403-94de-430c-ae9b-193d7c8a4275/indexes/healthai/browser